# 02 - Clean Tanqeeb Job Data

This notebook is the **cleaning** stage of the Tanqeeb pipeline.

- Reads ONLY the jobs waiting in the *pending raw queue*
  (`tanqeeb_pending_raw_queue.json`) -- i.e. jobs `01_scrape_Tanqeeb.ipynb`
  has scraped but that haven't been cleaned yet. Already-cleaned jobs are
  never reprocessed.
- Performs data cleaning only on that batch (column ordering / final
  schema enforcement happens later, in `03_transform_Tanqeeb.ipynb`):
  - Strips extra/duplicate whitespace from every text field.
  - Normalizes missing or empty values to "N/A".
  - Removes exact duplicate job entries (same `job_url`) within the batch.
- **Appends** the cleaned batch onto the historical cleaned dataset
  (`tanqeeb_jobs_cleaned.json`), instead of rebuilding it from scratch.
- Pushes the same cleaned batch onto a *pending cleaned queue* for
  `03_transform_Tanqeeb.ipynb`, then clears the raw queue it just consumed.


In [1]:
import json
import os
import pandas as pd


In [2]:
# ============================================================
# CONFIG
# ============================================================

# Input: produced (appended to) by 01_scrape_Tanqeeb.ipynb
RAW_PENDING_QUEUE = "../data/raw/tanqeeb_pending_raw_queue.json"

# Cumulative, historical cleaned dataset -- this notebook APPENDS to it
CLEANED_JSON_OUTPUT = "../data/interim/tanqeeb_jobs_cleaned.json"

# Output queue: consumed (and cleared) by 03_transform_Tanqeeb.ipynb
CLEANED_PENDING_QUEUE = "../data/interim/tanqeeb_pending_cleaned_queue.json"


In [3]:
def load_json_list(path):
    """Load a JSON file containing a list of job dicts; [] if missing."""
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_json_list(path, items):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)


## Load ONLY the new (not-yet-cleaned) jobs from the pending queue

In [4]:
raw_jobs = load_json_list(RAW_PENDING_QUEUE)
print("New raw jobs waiting to be cleaned:", len(raw_jobs))

df = pd.DataFrame(raw_jobs)
df.head()


New raw jobs waiting to be cleaned: 0


""


## Clean: strip extra whitespace from text fields

In [5]:
text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


## Clean: normalize missing/empty values to "N/A"

In [6]:
df = df.replace(
    to_replace=["", "none", "None", "null", "NULL", "nan", "NaN"],
    value="N/A",
)
df = df.fillna("N/A")


## Clean: remove exact duplicate job entries (same job_url) within this batch

In [7]:
before = len(df)

if "job_url" in df.columns:
    df = df.drop_duplicates(subset=["job_url"], keep="first")
else:
    df = df.drop_duplicates(keep="first")

after = len(df)
print(f"Removed {before - after} duplicate job(s). {after} unique jobs remain.")


Removed 0 duplicate job(s). 0 unique jobs remain.


## Append this batch to the historical cleaned dataset

In [8]:
cleaned_batch = df.to_dict(orient="records")

history_df = pd.DataFrame(load_json_list(CLEANED_JSON_OUTPUT))
combined_df = pd.concat([history_df, df], ignore_index=True)

# Idempotent: if this notebook is accidentally re-run on the same batch,
# duplicates against history are dropped rather than piling up.
if "job_url" in combined_df.columns and not combined_df.empty:
    combined_df = combined_df.drop_duplicates(subset=["job_url"], keep="last")

save_json_list(CLEANED_JSON_OUTPUT, combined_df.to_dict(orient="records"))

print(f"Historical cleaned dataset now has {len(combined_df)} job(s) "
      f"(+{len(cleaned_batch)} from this run).")


Historical cleaned dataset now has 10001 job(s) (+0 from this run).


## Hand off to 03_transform, then clear the raw queue we just consumed

In [9]:
pending_cleaned = load_json_list(CLEANED_PENDING_QUEUE)
pending_urls = {j.get("job_url") for j in pending_cleaned}

for job in cleaned_batch:
    if job.get("job_url") not in pending_urls:
        pending_cleaned.append(job)
        pending_urls.add(job.get("job_url"))

save_json_list(CLEANED_PENDING_QUEUE, pending_cleaned)

# These raw jobs have now been cleaned -- remove them from the raw queue
save_json_list(RAW_PENDING_QUEUE, [])

print("\n" + "=" * 60)
print("Jobs cleaned this run:", len(cleaned_batch))
print("Total jobs waiting to be transformed (pending queue):", len(pending_cleaned))
print("Historical cleaned dataset saved as:", CLEANED_JSON_OUTPUT)
print("Raw pending queue cleared.")
print("=" * 60)



Jobs cleaned this run: 0
Total jobs waiting to be transformed (pending queue): 0
Historical cleaned dataset saved as: ../data/interim/tanqeeb_jobs_cleaned.json
Raw pending queue cleared.
